In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

    

# Inital Notes
- Add seasons during feature engineering
- Add is_pre_covid done, is_movement, season done (One hot encoding)

In [40]:
def harmonize_promoter(df):
    """
    Builds era-consistent promoter features for MTC2026 Phase 3.

    Why this is needed:
    - Majestic Theatre has no promoter code at all (single AEG partnership,
      not tracked per-show), so it needs its own bucket.
    - Magic Stick's raw Promoter Code is NOT comparable across years:
        2019 code 2 = "React co-pro"   (an external co-promoter)
        2022+ code 2 = "In-House - Brie" (in-house staff)
      Same code number, opposite meaning. Using raw Promoter Code as a
      numeric/categorical feature would silently conflate these.
    - Promoter Name text is already era-consistent (it names the actual
      promoter/staffer rather than a code), so we harmonize off of that
      instead of Promoter Code.

    Adds three columns:
      - promoter_group   : coarse, cross-era comparable bucket (use this
                            as the model feature)
      - promoter_detail  : finer-grained label (individual in-house staff
                            kept separate) for exploratory/reporting use
      - is_in_house       : bool, True for any in-house-booked show
                            (Magic Stick in-house staff OR Majestic/AEG,
                            since AEG is MTC's contracted partner, not an
                            arm's-length external co-promoter)
    """
    df = df.copy()

    # promoter_detail: light text cleanup, no cross-era collapsing yet
    detail_map = {
        'Magic Stick in-house': 'In-House - Unspecified (2019)',
        'In-House - Zach': 'In-House - Zach',
        'In-House - Brie': 'In-House - Brie',
        'In-house - Tom': 'In-House - Tom',
        'Paxahau - Tom Pattyn': 'Paxahau/Movement',
        'Black Iris': 'Black Iris',
        'Good Show - Jason Berry': 'Good Show',
        'Rental event': 'Rental',
        'Party Store co-pro': 'Party Store',
        'Further Frequencies': 'Further Frequencies',
        'React co-pro': 'React (2019 only)',
    }

    def get_detail(row):
        if row['Venue'] == 'Majestic Theatre':
            return 'AEG Partner'
        return detail_map.get(row['Promoter Name'], 'Unknown/Other')

    df['promoter_detail'] = df.apply(get_detail, axis=1)

    # promoter_group: coarse bucket, safe to compare 2019 vs 2022+
    group_map = {
        'In-House - Unspecified (2019)': 'In-House',
        'In-House - Zach': 'In-House',
        'In-House - Brie': 'In-House',
        'In-House - Tom': 'In-House',
        'Paxahau/Movement': 'Paxahau/Movement',
        'Black Iris': 'Black Iris',
        'Good Show': 'Good Show',
        'Rental': 'Rental',
        'Party Store': 'Party Store',
        'Further Frequencies': 'Further Frequencies',
        'React (2019 only)': 'External Co-Pro (Other)',
        'AEG Partner': 'AEG Partner',
        'Unknown/Other': 'Unknown/Other',
    }
    df['promoter_group'] = df['promoter_detail'].map(group_map)

    in_house_groups = {'In-House', 'AEG Partner'}
    df['is_in_house'] = df['promoter_group'].isin(in_house_groups)

    return df

def create_season(df):
    df = df.copy()
    
    season_map = {
        12: 'Winter', 1: 'Winter', 2: 'Winter',
        3: 'Spring', 4: 'Spring', 5: 'Spring',
        6: 'Summer', 7: 'Summer', 8: 'Summer',
        9: 'Fall', 10: 'Fall', 11: 'Fall'
    }
    
    df['season'] = df['Date'].dt.month.map(season_map)
    
    return df


def add_is_movement(df, date_col='Date'):
    """
    Flags shows falling in the Movement Festival weekend window:
    the Thursday before through Memorial Day Monday itself (5-day window).

    Memorial Day is the last Monday of May, computed per-year rather than
    hardcoded, so this works automatically for any year in the dataset
    (including future years for forecasting).

    Note: 2023 has no shows at all in this window at either venue - that's
    a genuine data gap, not a bug in this function. It will correctly show
    zero is_movement=True rows for 2023.
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    years = df[date_col].dt.year.unique()
    windows = []
    for year in years:
        # last Monday of May: start from May 31 and walk back to Monday
        last_day_of_may = pd.Timestamp(year=year, month=5, day=31)
        offset = (last_day_of_may.dayofweek - 0) % 7  # 0 = Monday
        memorial_day = last_day_of_may - pd.Timedelta(days=offset)
        window_start = memorial_day - pd.Timedelta(days=4)  # Thursday before
        windows.append((window_start, memorial_day))

    def in_window(d):
        return any(start <= d <= end for start, end in windows)

    df['is_movement'] = df[date_col].apply(in_window)
    return df

In [45]:
def build_feature_matrix(df):
    """
    Builds the model-ready feature matrix. Categorical columns are
    one-hot encoded; booleans/numerics pass through as-is.

    Deliberately excludes:
      - promoter_detail (too granular / not cross-era comparable - use
        promoter_group instead)
      - Event/Artist Name, Promoter Name, Promoter Code, Source File,
        Data Notes (identifiers/free text, not predictive features)
    """
    feature_cols_categorical = ['Venue', 'Genre', 'season', 'promoter_group']
    feature_cols_passthrough = ['day_of_week_num', 'Month', 'post_covid',
                                 'is_movement', 'is_in_house']

    X_cat = pd.get_dummies(df[feature_cols_categorical], drop_first=False)
    X_num = df[feature_cols_passthrough].astype(int)
    X = pd.concat([X_num, X_cat], axis=1)

    return X

In [46]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


def time_series_cv(df, target_name, model_type='xgb', n_splits=5):
    """
    Runs TimeSeriesSplit cross-validation on the 2019-2025 training data.
    This is only for model selection / getting a trustworthy performance
    estimate - it does not touch the Jan-May 2026 or June-July 2026 holdouts.

    TimeSeriesSplit always trains on an earlier chunk and validates on a
    later chunk, expanding forward - so fold 1 trains on the earliest slice
    and validates on the next chunk, fold 2 trains on everything up through
    that point and validates on the next chunk after that, etc. This avoids
    the leakage a normal random-shuffle KFold would cause (letting the model
    see 2025 shows while predicting 2020 shows).
    """
    df_sorted = df.sort_values('Date').reset_index(drop=True)

    # drop rows with missing target for this specific metric
    df_sorted = df_sorted[df_sorted[target_name].notna()].reset_index(drop=True)

    X = build_feature_matrix(df_sorted)
    y = df_sorted[target_name]

    tscv = TimeSeriesSplit(n_splits=n_splits)

    fold_results = []
    for fold_i, (train_idx, val_idx) in enumerate(tscv.split(X), start=1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # align columns - a category present in one fold slice might be
        # absent in another, since fold sizes/eras differ
        X_train, X_val = X_train.align(X_val, join='outer', axis=1, fill_value=0)

        if model_type == 'xgb':
            from xgboost import XGBRegressor
            model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
        else:
            from sklearn.ensemble import RandomForestRegressor
            model = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42)

        model.fit(X_train, y_train)
        pred = model.predict(X_val)

        mae = mean_absolute_error(y_val, pred)
        rmse = np.sqrt(mean_squared_error(y_val, pred))
        r2 = r2_score(y_val, pred)

        fold_results.append({
            'fold': fold_i, 'n_train': len(train_idx), 'n_val': len(val_idx),
            'MAE': mae, 'RMSE': rmse, 'R2': r2
        })

    return pd.DataFrame(fold_results)

In [47]:
df = pd.read_csv('../data/processed/all_shows_fixed.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.columns
df['Day of Week'] = df['Date'].dt.dayofweek

df.rename(columns={'Day of Week': 'day_of_week_num'}, inplace=True)
df[['Date', 'day_of_week_num']]
df['Month'] = df['Date'].dt.month
df['post_covid'] = df['Date'].dt.year >= 2022

df = create_season(df)
df = harmonize_promoter(df)
df = add_is_movement(df)
df = df.drop(columns='Date_parsed')



In [48]:
X = build_feature_matrix(df)
print(X.shape)
# CV only on the historical training population (2019-2025) -
# Jan-May 2026 validation and June-July 2026 test stay untouched
train_pool = df[df['Date'] < '2026-01-01'].copy()

for target_name in ['Capacity Rate', 'Show Gross Proceeds']:
    for model_type in ['xgb', 'rf']:
        print(f"=== {target_name} | {model_type.upper()} ===")
        results = time_series_cv(train_pool, target_name, model_type=model_type, n_splits=5)
        print(results.to_string(index=False))
        print(f"  Mean R2 across folds: {results['R2'].mean():.4f}  (std: {results['R2'].std():.4f})")
        print(f"  Mean MAE across folds: {results['MAE'].mean():.4f}")
        print()

(844, 29)
=== Capacity Rate | XGB ===
 fold  n_train  n_val      MAE     RMSE        R2
    1      132    129 0.237967 0.296514 -0.253950
    2      261    129 0.265868 0.313759 -0.018004
    3      390    129 0.262168 0.317036 -0.066901
    4      519    129 0.241079 0.288322  0.022089
    5      648    129 0.247075 0.294257  0.129905
  Mean R2 across folds: -0.0374  (std: 0.1411)
  Mean MAE across folds: 0.2508

=== Capacity Rate | RF ===
 fold  n_train  n_val      MAE     RMSE        R2
    1      132    129 0.238359 0.300039 -0.283941
    2      261    129 0.275911 0.320634 -0.063107
    3      390    129 0.255105 0.308630 -0.011071
    4      519    129 0.241855 0.287087  0.030451
    5      648    129 0.246886 0.294724  0.127141
  Mean R2 across folds: -0.0401  (std: 0.1531)
  Mean MAE across folds: 0.2516

=== Show Gross Proceeds | XGB ===
 fold  n_train  n_val          MAE         RMSE        R2
    1      132    130 10151.824601 12537.666024 -0.413002
    2      262    130  92

In [35]:
df.head()

,Venue,Date,day_of_week_num,Event/Artist Name,Genre,Promoter Code,Promoter Name,Tickets Sold,Venue Capacity,Capacity Rate,...,Source Year (file),Source File,Data Notes,Month,post_covid,season,promoter_detail,promoter_group,is_in_house,is_movement
0,Majestic Theatre,2019-01-02,2,Noname,Hip-Hop/R&B,NaN,NaN,1092.0,1100.0,0.992727,...,2019,2019_show_results___metrics_2019_11_12_19.xlsx,NaN,1,False,Winter,AEG Partner,AEG Partner,True,False
1,Majestic Theatre,2019-01-05,5,Ookay (1),Electronic/Dance,NaN,NaN,658.0,1100.0,0.598182,...,2019,2019_show_results___metrics_2019_11_12_19.xlsx,NaN,1,False,Winter,AEG Partner,AEG Partner,True,False
2,Majestic Theatre,2019-01-24,3,Piff the Magic Dragon,Comedy/Other,NaN,NaN,325.0,400.0,0.812500,...,2019,2019_show_results___metrics_2019_11_12_19.xlsx,NaN,1,False,Winter,AEG Partner,AEG Partner,True,False
3,Magic Stick,2019-01-26,5,Will Sessions,Jazz/Blues/Soul,1.0,Magic Stick in-house,270.0,700.0,0.385714,...,2019,2019_show_results___metrics_2019_11_12_19.xlsx,NaN,1,False,Winter,In-House - Unspecified (2019),In-House,True,False
4,Majestic Theatre,2019-01-26,5,The Amity Affliction,Metal/Hard Rock,NaN,NaN,1100.0,1100.0,1.000000,...,2019,2019_show_results___metrics_2019_11_12_19.xlsx,NaN,1,False,Winter,AEG Partner,AEG Partner,True,False
